# AI-Generated Image Detection using ResNet18

This notebook develops a ResNet18-based binary image classifier to distinguish between REAL and AI-GENERATED images using the CIFAKE dataset.

This notebook develops and evaluates a ResNet18-based binary image classifier to distinguish between REAL and AI-GENERATED images using the CIFAKE dataset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Dataset Loading and Preparation

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from PIL import Image

In [ ]:
zip_path = "/content/drive/MyDrive/Intro_to_AI_Group_Project /dataset.zip"

print("ZIP exists in Drive:", os.path.exists(zip_path))

In [ ]:
!cp "/content/drive/MyDrive/Intro_to_AI_Group_Project /dataset.zip" "/content/cifake.zip"

In [ ]:
!unzip -q "/content/cifake.zip" -d "/content/CIFAKE"

In [ ]:
import os

print(os.listdir("/content/CIFAKE"))

In [ ]:
train_dir = "/content/CIFAKE/train"
test_dir = "/content/CIFAKE/test"

## Image Preprocessing and DataLoaders

In [ ]:
import os

print("Train classes:", os.listdir(train_dir))
print("Test classes:", os.listdir(test_dir))

for split_name, split_dir in [("Train", train_dir), ("Test", test_dir)]:
    print(f"\n{split_name}")

    for class_name in sorted(os.listdir(split_dir)):
        class_path = os.path.join(split_dir, class_name)
        if os.path.isdir(class_path):
            print(class_name, ":", len(os.listdir(class_path)))

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights

# Use the preprocessing expected by pretrained ResNet18
weights = ResNet18_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=test_transform
)

print("Class mapping:", train_dataset.class_to_idx)
print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

In [ ]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

## Stage 1 — Frozen-Feature Baseline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

images, labels = next(iter(train_loader))

# Undo ImageNet normalisation for display
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).numpy()
    img = std * img + mean
    img = np.clip(img, 0, 1)

    ax.imshow(img)
    ax.set_title("FAKE" if labels[i].item() == 0 else "REAL")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Freeze pretrained feature extractor for the baseline
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

print("Using device:", device)
print(model.fc)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [ ]:
num_epochs = 3

train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.4f}"
    )

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import numpy as np
import torch

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)
        predicted = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

        # Probability of FAKE class = class 0
        all_probabilities.extend(probabilities[:, 0].cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

# Convert labels so FAKE is treated as positive class:
# original: FAKE=0, REAL=1
# evaluation: FAKE=1, REAL=0
fake_true = (all_labels == 0).astype(int)
fake_pred = (all_predictions == 0).astype(int)

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(fake_true, fake_pred)
recall = recall_score(fake_true, fake_pred)
f1 = f1_score(fake_true, fake_pred)
roc_auc = roc_auc_score(fake_true, all_probabilities)

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"FAKE Precision: {precision:.4f}")
print(f"FAKE Recall:    {recall:.4f}")
print(f"FAKE F1-score:  {f1:.4f}")
print(f"ROC-AUC:        {roc_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["FAKE", "REAL"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

In [ ]:
baseline_path = "/content/drive/MyDrive/Intro_to_AI_Group_Project /resnet18_baseline.pth"

torch.save(model.state_dict(), baseline_path)

print("Baseline saved to:", baseline_path)

## Stage 2 — Fine-Tuning ResNet18

In [ ]:
from torch.utils.data import Subset
import torch

# Separate dataset objects so validation does NOT use training augmentation
full_train_aug = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

full_train_eval = datasets.ImageFolder(
    train_dir,
    transform=test_transform
)

generator = torch.Generator().manual_seed(42)

indices = torch.randperm(
    len(full_train_aug),
    generator=generator
).tolist()

train_size = 90000

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_subset = Subset(full_train_aug, train_indices)
val_subset = Subset(full_train_eval, val_indices)

train_loader = DataLoader(
    train_subset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))

In [ ]:
# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the final ResNet block
for param in model.layer4.parameters():
    param.requires_grad = True

# Keep classifier trainable
for param in model.fc.parameters():
    param.requires_grad = True

In [ ]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    # ---- Training ----
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= train_total
    train_accuracy = train_correct / train_total

    # ---- Validation ----
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= val_total
    val_accuracy = val_correct / val_total

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

In [ ]:
import os
import torch

model_dir = "/content/drive/MyDrive/Intro_to_AI_Group_Project /models"
os.makedirs(model_dir, exist_ok=True)

finetuned_path = os.path.join(model_dir, "resnet18_finetuned.pth")

torch.save(model.state_dict(), finetuned_path)

print("Saved to:", finetuned_path)

## Final Test Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import numpy as np

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)
        predicted = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

        # FAKE is class 0
        all_probabilities.extend(probabilities[:, 0].cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

# Treat FAKE as the positive class for evaluation
fake_true = (all_labels == 0).astype(int)
fake_pred = (all_predictions == 0).astype(int)

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(fake_true, fake_pred)
recall = recall_score(fake_true, fake_pred)
f1 = f1_score(fake_true, fake_pred)
roc_auc = roc_auc_score(fake_true, all_probabilities)

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"FAKE Precision: {precision:.4f}")
print(f"FAKE Recall:    {recall:.4f}")
print(f"FAKE F1-score:  {f1:.4f}")
print(f"ROC-AUC:        {roc_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["FAKE", "REAL"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(all_labels, all_predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["FAKE", "REAL"]
)

disp.plot()
plt.title("ResNet18 Confusion Matrix on CIFAKE Test Set")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(fake_true, all_probabilities)
roc_auc_value = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_value:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - ResNet18")
plt.legend()
plt.show()

## Fine-Tuning Training History

In [ ]:
# Fine-tuning metrics recorded from the three training epochs above.
# These values are used only to reproduce the training/validation plots.

fine_tune_train_acc = [0.9430, 0.9710, 0.9797]
fine_tune_val_acc = [0.9618, 0.9637, 0.9697]

fine_tune_train_loss = [0.1488, 0.0768, 0.0544]
fine_tune_val_loss = [0.1048, 0.0933, 0.0900]

epochs = [1, 2, 3]

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(epochs, fine_tune_train_acc, marker="o", label="Training Accuracy")
plt.plot(epochs, fine_tune_val_acc, marker="o", label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ResNet18 Fine-Tuning Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(epochs, fine_tune_train_loss, marker="o", label="Training Loss")
plt.plot(epochs, fine_tune_val_loss, marker="o", label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet18 Fine-Tuning Loss")
plt.legend()
plt.show()

## Inference and Example Predictions

In [ ]:
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
from PIL import Image

def predict_image(filepath, model, device):
    image = Image.open(filepath).convert("RGB")

    image_tensor = inference_transform(image)
    image_tensor = image_tensor.unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    fake_probability = probabilities[0].item()
    real_probability = probabilities[1].item()

    if fake_probability > real_probability:
        prediction = "AI-GENERATED"
        confidence = fake_probability
    else:
        prediction = "REAL"
        confidence = real_probability

    return {
        "prediction": prediction,
        "confidence": confidence,
        "fake_probability": fake_probability,
        "real_probability": real_probability
    }

In [ ]:
sample_path = "/content/CIFAKE/test/FAKE/10 (10).jpg"

result = predict_image(
    sample_path,
    model,
    device
)

print(result)

In [ ]:
real_test_dir = "/content/CIFAKE/test/REAL"
real_files = os.listdir(real_test_dir)

print(real_files[:10])

real_sample_path = os.path.join(real_test_dir, real_files[0])

result = predict_image(
    real_sample_path,
    model,
    device
)

print(result)

## Saving the Final Model

In [ ]:
checkpoint_path = os.path.join(
    model_dir,
    "resnet18_finetuned_checkpoint.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_to_idx": {"FAKE": 0, "REAL": 1},
        "input_size": (224, 224),
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    checkpoint_path
)

print("Checkpoint saved to:", checkpoint_path)

In [ ]:
import random
import matplotlib.pyplot as plt

def show_predictions(folder, true_label, n=6):
    files = random.sample(os.listdir(folder), n)

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))

    for ax, filename in zip(axes.flat, files):
        path = os.path.join(folder, filename)

        result = predict_image(path, model, device)
        image = Image.open(path).convert("RGB")

        ax.imshow(image)
        ax.set_title(
            f"True: {true_label}\n"
            f"Pred: {result['prediction']}\n"
            f"Conf: {result['confidence']:.2%}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_predictions("/content/CIFAKE/test/FAKE", "FAKE")

In [ ]:
show_predictions("/content/CIFAKE/test/REAL", "REAL")

In [ ]:
ui_model_path = os.path.join(
    model_dir,
    "rai_resnet18_cifake_final.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_to_idx": {
            "FAKE": 0,
            "REAL": 1
        },
        "input_size": 224,
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    ui_model_path
)

print("UI-ready model saved:", ui_model_path)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

loaded_model = resnet18(weights=None)

loaded_model.fc = nn.Linear(
    loaded_model.fc.in_features,
    2
)

checkpoint = torch.load(
    ui_model_path,
    map_location=device
)

loaded_model.load_state_dict(
    checkpoint["model_state_dict"]
)

loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Model reloaded successfully.")

In [ ]:
result = predict_image(
    "/content/CIFAKE/test/FAKE/10 (10).jpg",
    loaded_model,
    device
)

print(result)

In [ ]:
def classify_uploaded_image(filepath):
    result = predict_image(
        filepath,
        loaded_model,
        device
    )

    return {
        "label": result["prediction"],
        "confidence": round(result["confidence"], 4),
        "ai_probability": round(result["fake_probability"], 4),
        "real_probability": round(result["real_probability"], 4)
    }

In [ ]:
classify_uploaded_image(
    "/content/CIFAKE/test/FAKE/10 (10).jpg"
)

## Grad-CAM Model Interpretation

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

gradients = None
activations = None

def save_gradient(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0]

def save_activation(module, input, output):
    global activations
    activations = output

target_layer = loaded_model.layer4[-1].conv2

forward_handle = target_layer.register_forward_hook(save_activation)
backward_handle = target_layer.register_full_backward_hook(save_gradient)

In [ ]:
def generate_gradcam(filepath, model, device):
    global gradients, activations

    original_image = Image.open(filepath).convert("RGB")

    image_tensor = inference_transform(original_image)
    image_tensor = image_tensor.unsqueeze(0).to(device)

    model.eval()
    model.zero_grad()

    outputs = model(image_tensor)

    predicted_class = outputs.argmax(dim=1).item()

    score = outputs[0, predicted_class]
    score.backward()

    pooled_gradients = torch.mean(
        gradients,
        dim=[0, 2, 3]
    )

    activation_map = activations[0].detach().clone()

    for i in range(activation_map.shape[0]):
        activation_map[i] *= pooled_gradients[i]

    heatmap = torch.mean(activation_map, dim=0)

    heatmap = F.relu(heatmap)

    heatmap /= torch.max(heatmap) + 1e-8

    heatmap = heatmap.cpu().numpy()

    return original_image, heatmap, predicted_class

In [ ]:
def show_gradcam(filepath, model, device):
    original_image, heatmap, predicted_class = generate_gradcam(
        filepath,
        model,
        device
    )

    label = "AI-GENERATED" if predicted_class == 0 else "REAL"

    heatmap_resized = Image.fromarray(
        np.uint8(255 * heatmap)
    ).resize(original_image.size)

    heatmap_resized = np.array(heatmap_resized) / 255.0

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(original_image)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_resized, cmap="jet")
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(original_image)
    plt.imshow(
        heatmap_resized,
        cmap="jet",
        alpha=0.45
    )
    plt.title(f"Prediction: {label}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_gradcam(
    "/content/CIFAKE/test/FAKE/10 (10).jpg",
    loaded_model,
    device
)

In [ ]:
show_gradcam(
    real_sample_path,
    loaded_model,
    device
)

## Model Deployment and Gradio UI Integration

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet18
from PIL import Image, ImageOps

In [ ]:
def predict_cnn(image):
    image = ImageOps.exif_transpose(image).convert("RGB")

    image_tensor = inference_transform(image)
    image_tensor = image_tensor.unsqueeze(0).to(device)

    loaded_model.eval()

    with torch.no_grad():
        outputs = loaded_model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    return {
        "FAKE": probabilities[0].item(),
        "REAL": probabilities[1].item()
    }

## Final Model Summary

The fine-tuned ResNet18 achieved 96.80% accuracy, a weighted F1-score of 0.9680 and a ROC-AUC of 0.9957 on the untouched 20,000-image CIFAKE test set. Fine-tuning layer4 and the classification layer improved test accuracy by 8.28 percentage points over the frozen-feature baseline. The final checkpoint was saved and successfully reloaded for deployment and Gradio integration.